# 基于 MindNLP 实现 BERT 情感分类 (SST-2)

本实验基于 MindNLP 套件，使用预训练的 BERT 模型在 GLUE 基准数据集的 SST-2（Stanford Sentiment Treebank）任务上进行微调（Fine-tuning），实现对电影评论的情感分类（正面/负面）。


# BERT 模型原理简介

### (1)  BERT 概述
BERT (Bidirectional Encoder Representations from Transformers) 是由 Google 在 2018 年提出的预训练语言模型。它的出现是自然语言处理 (NLP) 领域的里程碑事件，大幅刷新了当时 11 项 NLP 任务的 SOTA 记录。

BERT 的核心架构基于 Transformer 的 **Encoder (编码器)** 部分。与传统的单向语言模型（如 GPT 从左到右）或浅层双向模型（如 Bi-LSTM）不同，BERT 利用 **Masked Language Model (MLM)** 预训练目标，使其能够真正地同时从上下文两个方向学习深层的语义表示。

![bert模型结构](images\bert.png)

### (2)  核心特性
- **双向性 (Bidirectionality)**: BERT 在处理每个词时，都能同时看到它之前和之后的词，从而更准确地理解语境（Context）。
- **Transformer 编码器**: 采用多层 Self-Attention 机制，并行计算能力强，能捕捉长距离依赖。
- **预训练任务**:
    1.  **MLM (掩码语言模型)**: 随机遮盖输入中的部分 Token，让模型去预测它们。
    2.  **NSP (下一句预测)**: 判断两个句子是否是连续的，帮助模型理解句子间的关系。

### (3)  本实验使用的模型
在本实验中，我们使用 **`bert-base-uncased`** 模型：
- **Base**: 代表模型规模（12层 Transformer Block，768 隐藏层维度，12 个 Attention Heads，约 1.1 亿参数）。
- **Uncased**: 代表在预训练前将所有文本转换为小写（即不区分大小写）。

### (4)  BERT 在文本分类中的应用
在 SST-2 情感分类任务中，我们利用 BERT 的特殊标记 `[CLS]`：
1.  BERT 在输入序列的开头自动添加一个 `[CLS]` 标记。
2.  经过 12 层 Transformer 处理后，`[CLS]` 位置输出的向量被视为整个句子的语义表示（Sentence Representation）。
3.  我们在 `[CLS]` 向量之上添加一个简单的全连接层（Classifier），将维度从 768 映射到 2（Positive/Negative），即可完成分类任务。

In [1]:
# !pip install mindnlp mindspore evaluate tqdm

## 1. 实验环境
- MindSpore 版本:  2.7.0
- MindNLP 版本:  0.5.1
- 硬件环境: Ascend/GPU

## 2. 导入库与环境设置

设置 MindSpore 运行模式。为了保证在不同硬件上的兼容性以及与 HuggingFace 风格 Trainer 的适配，推荐使用 `PYNATIVE_MODE`（动态图模式）。

In [ ]:
import os
import sys
import warnings
import numpy as np
import mindspore
from mindspore import context

# 设置运行模式为 PYNATIVE_MODE (推荐)
context.set_context(mode=context.PYNATIVE_MODE, device_target="Ascend")

# 屏蔽底层繁杂日志
os.environ['GLOG_v'] = '3'
warnings.filterwarnings("ignore")

print(f"MindSpore Version: {mindspore.__version__}")

## 3. 数据集加载

使用 MindNLP 的 `load_dataset` 接口下载并加载 GLUE 基准中的 SST-2 数据集。SST-2 是一个二分类的情感分析数据集。

In [ ]:
from mindnlp.dataset import load_dataset

TASK = "sst2"
MODEL_CHECKPOINT = "bert-base-uncased"

print(f"正在加载 GLUE/{TASK} 数据集...")
dataset_dict = load_dataset("glue", TASK)
print(f"训练集样本数: {dataset_dict['train'].get_dataset_size()}")
print(f"验证集样本数: {dataset_dict['validation'].get_dataset_size()}")

## 4. 数据预处理

我们需要对文本进行 Tokenize（分词）并转换为模型可接受的 Tensor 格式。

**注意**：为了确保 `Trainer` 的兼容性，我们将流式数据集处理为内存中的 List 格式，并对 numpy 类型的数据进行显式转换。

In [ ]:
from mindnlp.transformers import AutoTokenizer
from tqdm import tqdm

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def process_dataset_to_list(dataset, tokenizer, max_seq_len=128):
    """
    将 MindSpore 数据集进行分词处理，并转换为内存列表以适配 Trainer
    """
    # 定义分词逻辑
    def tokenize(text):
        # 类型安全检查：确保输入为字符串
        if isinstance(text, np.ndarray):
            text = str(text.item())
        if isinstance(text, bytes):
            text = text.decode('utf-8')
            
        tokenized = tokenizer(text, padding='max_length', truncation=True, max_length=max_seq_len)
        return tokenized['input_ids'], tokenized['attention_mask'], tokenized['token_type_ids']

    # 识别文本列名
    col_names = dataset.column_names
    text_col = 'sentence' if 'sentence' in col_names else col_names[0]
    
    # 使用 map 操作进行分词
    dataset = dataset.map(
        operations=tokenize, 
        input_columns=text_col, 
        output_columns=['input_ids', 'attention_mask', 'token_type_ids']
    )
    
    # Label 类型转换
    if 'label' in col_names:
        dataset = dataset.map(operations=lambda x: x.astype("int32"), input_columns="label")

    # 转换为 Python List
    data_list = []
    iterator = dataset.create_dict_iterator(output_numpy=True)
    
    print("正在转换数据格式...")
    for item in tqdm(iterator):
        # 重命名 label 为 labels 以匹配 HF Trainer 标准
        if 'label' in item:
            item['labels'] = item.pop('label')
        data_list.append(item)
        
    return data_list

# 执行预处理
print("处理训练集...")
train_dataset = process_dataset_to_list(dataset_dict['train'], tokenizer)
print("处理验证集...")
eval_dataset = process_dataset_to_list(dataset_dict['validation'], tokenizer)

## 5. 模型构建与评估指标

加载预训练的 BERT 模型，并定义评估指标（Accuracy）。

In [ ]:
from mindnlp.transformers import AutoModelForSequenceClassification
import evaluate

# 加载 BERT 分类模型 (num_labels=2)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)

# 定义评估函数
def compute_metrics(eval_pred):
    try:
        metric = evaluate.load("glue", TASK)
    except Exception:
        # 离线备用方案
        def simple_acc(predictions, references):
            return {"accuracy": (predictions == references).mean()}
        metric = type("Metric", (), {"compute": lambda self, predictions, references: simple_acc(predictions, references)})()
        
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

## 6. 模型训练

使用 `TrainingArguments` 配置超参数，并使用 `Trainer` 启动训练流程。

In [ ]:
from mindnlp.transformers import Trainer, TrainingArguments

BATCH_SIZE = 32

training_args = TrainingArguments(
    output_dir=f"output_{TASK}",
    eval_strategy="epoch",       # 每个 epoch 结束后评估
    save_strategy="epoch",       # 每个 epoch 结束后保存
    logging_steps=10,            # 日志打印频率
    learning_rate=2e-5,          # 学习率
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=3,          # 训练轮数
    save_total_limit=1,          # 只保留最新的模型
    load_best_model_at_end=True, # 训练结束加载最优模型
    metric_for_best_model="accuracy"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset, 
    eval_dataset=eval_dataset,   
    compute_metrics=compute_metrics,
)

print("开始训练...")
trainer.train()

## 7. 模型保存

将微调后的模型权重和分词器保存到本地，以便后续推理使用。

In [ ]:
save_path = "./bert_sst2_finetuned"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"模型已保存至: {save_path}")

## 8. 模型推理

使用 `pipeline` 高阶接口加载保存的模型，对新文本进行情感预测。

In [ ]:
from mindnlp.transformers import pipeline

# 加载保存的模型进行推理
classifier = pipeline("sentiment-analysis", model=save_path, tokenizer=save_path, top_k=None)

# 测试用例
test_sentences = [
    "I absolutely love this movie, it's fantastic!", 
    "The plot was boring and the acting was terrible.",
    "It was okay, not great but not bad."
]

print("-" * 30)
print("推理结果展示:")
print("-" * 30)

for text in test_sentences:
    results = classifier(text)
    # 解析结果
    scores = results[0]
    # 获取最高分标签
    best_result = max(scores, key=lambda x: x['score'])
    label = "Positive" if best_result['label'] == 'LABEL_1' else "Negative"
    
    print(f"文本: {text}")
    print(f"预测: {label} (置信度: {best_result['score']:.4f})\n")